# LDA Topic Modeling
MACS 30113 Final Project — Analyzing and Modeling part, Anyi Li

Runs Latent Dirichlet Allocation (LDA) separately on pre-COVID and post-COVID Reddit data to identify whether discussion themes shifted around the pandemic onset. Reads from Nana's preprocessed Parquet files.

In [1]:
%%configure -f
{
    "conf": {
        "spark.pyspark.python": "python3",
        "spark.pyspark.virtualenv.enabled": "true",
        "spark.pyspark.virtualenv.type": "native",
        "spark.pyspark.virtualenv.bin.path": "/usr/bin/virtualenv"
    }
}

In [2]:
spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
2,application_1779939149554_0004,pyspark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 1. Load Data

In [3]:
posts_nlp = spark.read.parquet("s3://30113-final-project/reddit/posts_nlp/")
comments_nlp = spark.read.parquet("s3://30113-final-project/reddit/comments_nlp/")

common_cols = list(set(posts_nlp.columns) & set(comments_nlp.columns))
reddit_df = posts_nlp.select(common_cols).union(comments_nlp.select(common_cols))

print("Total records:", reddit_df.count())
reddit_df.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total records: 4538
root
 |-- subreddit: string (nullable = true)
 |-- filtered_tokens: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- year: integer (nullable = true)
 |-- post_id: string (nullable = true)
 |-- score: long (nullable = true)
 |-- tokens: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- num_tokens: integer (nullable = true)
 |-- source: string (nullable = true)
 |-- clean_text: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- month: integer (nullable = true)
 |-- created_utc: long (nullable = true)

## 2. Define COVID Event Windows

In [4]:
from pyspark.sql.functions import col, when, lit

reddit_df = reddit_df.withColumn(
    "period",
    when(
        (col("year") < 2020) | ((col("year") == 2020) & (col("month") < 3)),
        lit("pre_covid")
    ).otherwise(lit("post_covid"))
)

reddit_df.groupBy("period").count().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+-----+
|    period|count|
+----------+-----+
|post_covid| 4538|
+----------+-----+

## 3. LDA Topic Modeling
LDA assumes each document is a mixture of topics, and each topic is a distribution over words. CountVectorizer converts token lists into term-frequency vectors that LDA requires as input.

Parameters:
- `k=5`: number of topics
- `maxIter=10`: EM algorithm iterations
- `vocabSize=5000`: vocabulary cap
- `minDF=5`: ignore terms appearing in fewer than 5 documents

In [7]:
from pyspark.ml.feature import CountVectorizer
from pyspark.ml.clustering import LDA

NUM_TOPICS = 5
MAX_ITER = 10
MAX_VOCAB = 5000
MIN_DF = 2  # lowered from 5 to 2 for smaller dataset

def run_lda(df, period_label):
    print(f"\nRunning LDA: {period_label}")

    subset = df.filter(col("period") == period_label)

    # Convert token lists to sparse term-frequency vectors
    cv = CountVectorizer(
        inputCol="filtered_tokens",
        outputCol="features",
        vocabSize=MAX_VOCAB,
        minDF=MIN_DF
    )
    cv_model = cv.fit(subset)
    vectorized = cv_model.transform(subset)
    vocab = cv_model.vocabulary

    # Fit LDA — distributed across cluster nodes
    lda = LDA(k=NUM_TOPICS, maxIter=MAX_ITER, seed=42)
    lda_model = lda.fit(vectorized)

    # Print top 10 words per topic
    print(f"Top words per topic ({period_label}):")
    topics = lda_model.describeTopics(maxTermsPerTopic=10)
    for row in topics.collect():
        topic_words = [vocab[i] for i in row['termIndices']]
        print(f"  Topic {row['topic']}: {', '.join(topic_words)}")

    return lda_model, cv_model, vectorized

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
print("Pre-COVID count:", reddit_df.filter(col("period") == "pre_covid").count())
print("Post-COVID count:", reddit_df.filter(col("period") == "post_covid").count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Pre-COVID count: 0
Post-COVID count: 4538

# IMPORTANT NOTE: Uncomment pre-COVID parts with full dataset which include pre-COVID data. --Anyi

In [12]:
# Pre-COVID topic model
# lda_pre, cv_pre, vec_pre = run_lda(reddit_df, "pre_covid")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
# Post-COVID topic model
lda_post, cv_post, vec_post = run_lda(reddit_df, "post_covid")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


Running LDA: post_covid
Top words per topic (post_covid):
  Topic 0: im, thank, dont, sleep, time, really, think, one, got, day
  Topic 1: im, like, breath, dont, anxiety, ive, take, time, also, parents
  Topic 2: im, like, dont, feel, get, know, people, really, time, even
  Topic 3: dont, feel, know, even, like, get, day, cant, every, im
  Topic 4: im, ive, feel, help, someone, never, like, know, time, thanks

## 4. Topic Distribution by Subreddit
Assigns each document its dominant topic, then aggregates by subreddit and period to show which topics dominate in each community.

# IMPORTANT NOTE: Uncomment pre-COVID parts with full dataset which include pre-COVID data. --Anyi

In [11]:
from pyspark.sql.functions import udf, array_max
from pyspark.sql.types import IntegerType
import numpy as np

# UDF to extract the dominant topic index from LDA's topic distribution vector
dominant_topic_udf = udf(lambda v: int(np.argmax(v)), IntegerType())

# Note: pre-COVID data not yet available in current test batch
# Full comparison will be run with complete dataset

# Assign dominant topic to each document in pre period
#pre_with_topics = lda_pre.transform(vec_pre).withColumn(
#    "dominant_topic", dominant_topic_udf(col("topicDistribution"))
#)

# Assign dominant topic to each document in post period
post_with_topics = lda_post.transform(vec_post).withColumn(
    "dominant_topic", dominant_topic_udf(col("topicDistribution"))
)

# print("Pre-COVID: dominant topic distribution by subreddit")
# pre_with_topics.groupBy("subreddit", "dominant_topic").count() \
#    .orderBy("subreddit", "dominant_topic").show()

print("Post-COVID: dominant topic distribution by subreddit")
post_with_topics.groupBy("subreddit", "dominant_topic").count() \
    .orderBy("subreddit", "dominant_topic").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Post-COVID: dominant topic distribution by subreddit
+------------+--------------+-----+
|   subreddit|dominant_topic|count|
+------------+--------------+-----+
|     anxiety|             0|   52|
|     anxiety|             1|   24|
|     anxiety|             2| 1557|
|     anxiety|             3|   16|
|     anxiety|             4|   31|
|  depression|             0|   54|
|  depression|             1|    9|
|  depression|             2| 1830|
|  depression|             3|   21|
|  depression|             4|   21|
|mentalhealth|             0|   39|
|mentalhealth|             1|    6|
|mentalhealth|             2|  849|
|mentalhealth|             3|    6|
|mentalhealth|             4|   23|
+------------+--------------+-----+

## 5. Save Topic-Labeled Data to S3

# IMPORTANT NOTE: Uncomment pre-COVID parts with full dataset which include pre-COVID data. --Anyi

In [13]:
#pre_with_topics.write.mode("overwrite").parquet(
#    "s3://30113-final-project/results/topics_pre_covid/"
#)
post_with_topics.write.mode("overwrite").parquet(
    "s3://30113-final-project/results/topics_post_covid/"
)
print("Topic-labeled data saved.")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Topic-labeled data saved.